In [ ]:
# GUIDELINES
# 1. Install pip if you don't have it: https://pip.pypa.io/en/stable/installation/
# 2. Run "pip install pandas requests musicbrainzngs discogs-client" in your terminal
# 3. Create a Discogs account if you haven't: https://www.discogs.com/
# 4. Go to https://www.discogs.com/settings/developers
# 5. Click "Generate Token"
# 6. Copy the token where it sayd "Your tokens" to DISCOGS_TOKEN (as a string)
# 7. Change SHELF_NAME to the shelf you are using
# 8. Download your catalogged data (title, artist name) as a .csv file and put it in this directory
# NOTE Make sure the columns for artist and release title are "Artist" and "Title" (as they are on the Google sheet)
# 9. Change the file name where it says "Load CSV" to your file
# You should have changed three things in the script below. Now go ahead and run it!

In [ ]:
import pandas as pd
import musicbrainzngs
import requests
import discogs_client
import time

# Your tokens
DISCOGS_TOKEN = "yourtoken" # CHANGE THIS

SHELF_NAME = "yourshelf" # CHANGE THIS

# Setup APIs
discogs = discogs_client.Client("AlbumInfoFetcher/1.0", user_token=DISCOGS_TOKEN)

# Load CSV
df = pd.read_csv("yourfile.csv") # CHANGE THIS

# Columns to add
df["Genre(s)"] = None
df["Country"] = None
df["Similar Artists"] = None
df["Location (shelf)"] = None
df["Album Cover URL"] = None

def get_musicbrainz_data(artist, album):
    try:
        result = musicbrainzngs.search_releases(artist=artist, release=album, limit=1)
        if result['release-list']:
            release = result['release-list'][0]
            year = release.get("date", "").split("-")[0] if "date" in release else None
            country = release.get("country", None)
            return year, country
    except:
        return None, None
    return None, None

def get_lastfm_data(artist, album):
    try:
        url = f"http://ws.audioscrobbler.com/2.0/?method=album.getinfo&api_key={LASTFM_API_KEY}&artist={artist}&album={album}&format=json"
        resp = requests.get(url)
        data = resp.json().get("album", {})
        genre = None
        similar = None
        cover = None

        if "tags" in data and "tag" in data["tags"]:
            genre = ", ".join(tag["name"] for tag in data["tags"]["tag"])

        if "image" in data:
            cover = next((img["#text"] for img in data["image"] if img["size"] == "extralarge"), None)

        # Similar artists from artist info
        artist_url = f"http://ws.audioscrobbler.com/2.0/?method=artist.getsimilar&artist={artist}&api_key={LASTFM_API_KEY}&format=json"
        sim_resp = requests.get(artist_url).json()
        similar = ", ".join(a["name"] for a in sim_resp.get("similarartists", {}).get("artist", [])[:5])

        return cover, similar
    except:
        return None, None

def get_discogs_album_info(artist, album):
    try:
        query = f"{album} {artist}"
        results = discogs.search(query, type='release')
        
        if not results:
            print(f"❌ No Discogs result for: {album} by {artist}")
            return None, None, None, None

        release = results[0]  # First result is usually best

        year = getattr(release, 'year', None)
        genres = ', '.join(release.genres) if hasattr(release, 'genres') else None
        styles = ', '.join(release.styles) if hasattr(release, 'styles') else None
        cover_url = release.thumb if hasattr(release, 'thumb') else None

        return year, genres, styles, cover_url

    except Exception as e:
        return None, None, None, None
    
def get_discogs_album_info(artist, album, retries=3):
    import time
    for attempt in range(retries):
        try:
            query = f"{album} {artist}"
            results = discogs.search(query, type='release')
            if not results:
                return None, None, None, None, None

            release = results[0]
            year = getattr(release, 'year', None)
            genres = ', '.join(release.genres) if hasattr(release, 'genres') else None
            styles = ', '.join(release.styles) if hasattr(release, 'styles') else None
            cover_url = release.thumb if hasattr(release, 'thumb') else None
            country = getattr(release, 'country', None)

            time.sleep(1.2)  # rate limiting
            return year, genres, styles, cover_url, country
        except Exception as e:
            print(f"⚠️ Discogs attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)
    return None, None, None, None, None

# Main loop
for idx, row in df.iterrows():
    artist = row["Artist"]
    album = row["Title"]

    print(f"🔍 Processing: {album} by {artist}")

    year, a_country = get_musicbrainz_data(artist, album)
    cover, similar = get_lastfm_data(artist, album)

    # Discogs fallback if year or genre missing
    d_year, d_genre, d_styles, d_cover, d_country = get_discogs_album_info(artist, album)
    country = d_country
    if not country: country = a_country
    if not year: year = d_year
    genre = d_styles
    if not genre: genre = d_genre
    if not cover: cover = d_cover
    df.at[idx, "Genre(s)"] = genre

    df.at[idx, "Release Year"] = year
    df.at[idx, "Country"] = country
    df.at[idx, "Album Cover URL"] = cover
    df.at[idx, "Similar Artists"] = similar
    time.sleep(1) 
    
df["Location (shelf)"] = SHELF_NAME
df.to_csv("updated_albums.csv", index=False) # may want to change this
df